# Тема 7 — Тюнинг vLLM: двигаем саму кривую

**Вопрос темы.** Ваша рабочая точка из Topic 06 (T4, волна 64, ~460 ток/с) — максимум
конфигурации по умолчанию. А если другая даст на 20% больше токенов при том же SLO?
Вы не знаете — потому что кривую ещё не двигали.

**Что меняется:** у кривой насыщения есть ручки — `--max-num-seqs`, `--gpu-memory-utilization`,
`--max-model-len`, chunked prefill. Каждая управляет механизмом из Topic 06 (batching,
KV-бюджет, интерференция prefill) — см. картинку и карту ниже.

**Лаба, шесть шагов:** (1) таблица «ручка → ожидание → риск» ДО измерений;
(2) план из 5 конфигураций, замороженный ДО sweep; (3) все 5 через волны 8/32/64/128
с одним журналом; (4) Парето-фронт «скорость против хвоста», выбор под SLO;
(5) заморозка с прогнозом + вердикт held-out; (6) stretch: краевой тест и план отката.

**Правила:** неудачная конфигурация — строка в таблице, не вычёркиваемый эксперимент;
выбор после увиденного held-out — подгонка, не валидация; профиль без плана отката —
ставка без выхода. Прецедент — Round 03: FLASH_ATTN + `--max-num-seqs 64` дал +2.3%,
а быстрый Triton-кандидат остался отвергнутым из-за quality gate. Этот стандарт — ваш.

Профили: **mock** (CPU, минуты, объявленная синтетика) и **gpu** (T4, instructor).
Числа mock — не измерения vLLM. Конверт: 2 контактных часа + 3–5 самостоятельных.

In [ ]:
# Сквозной мотив темы одной картинкой: кривая насыщения и её ручки.
# Схема (объявлено): форма кривой и подписи — обучение, НЕ измерение этого ноутбука.
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд (Colab/Jupyter): инлайн-бэкенд
except NameError:
    matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# единый стиль всех графиков темы: шрифты, сетка, палитра
plt.rcParams.update({
    'figure.dpi': 110, 'font.size': 11, 'axes.titlesize': 12.5, 'axes.labelsize': 11,
    'axes.grid': True, 'grid.alpha': 0.3,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.prop_cycle': plt.cycler(color=['#2563eb', '#d97706', '#16a34a', '#dc2626', '#7c3aed']),
})

_xs = list(range(1, 141))
_thr = [560 * x / (x + 13.9) for x in _xs]  # схематичная кривая насыщения (не измерение)
fig, (ax, axb) = plt.subplots(
    2, 1, figsize=(8.4, 5.4), gridspec_kw=dict(height_ratios=[3.1, 1.15]))
ax.plot(_xs, _thr, lw=2.5, color='#2563eb')
ax.axvspan(64, 140, color='#f59e0b', alpha=0.10)
ax.text(104, 245, 'зона очередей: лишние клиенты\nпокупают очередь, а не токены',
        ha='center', fontsize=9.5, color='#b45309')
ax.scatter([64], [460], s=60, color='#dc2626', zorder=5)
ax.annotate('рабочая точка Topic 06\n(волна 64, ~460 ток/с)', xy=(64, 460), xytext=(80, 330),
            fontsize=9.5, arrowprops=dict(arrowstyle='->', color='#dc2626'))
ax.set_xlabel('параллельные клиенты (волна)')
ax.set_ylabel('throughput, ток/с')
ax.set_title('Кривая насыщения и её ручки — схема, не измерение')
ax.set_xlim(0, 140); ax.set_ylim(0, 620)

axb.axis('off'); axb.set_xlim(0, 1); axb.set_ylim(0, 1)
_knobs = [('--max-num-seqs', 'потолок batch decode'),
          ('--gpu-memory-utilization', 'запас KV-пула'),
          ('--max-model-len', 'планирование worst-case'),
          ('chunked prefill', 'чужой prefill не глушит decode')]
for i, (name, effect) in enumerate(_knobs):
    x0 = 0.015 + i * 0.248
    axb.add_patch(mpatches.FancyBboxPatch((x0, 0.18), 0.23, 0.66,
                  boxstyle='round,pad=0.012', fc='#f8fafc', ec='#2563eb', lw=1.3))
    axb.text(x0 + 0.115, 0.62, name, ha='center', va='center', fontsize=9, family='monospace')
    axb.text(x0 + 0.115, 0.30, effect, ha='center', va='center', fontsize=8.2, color='#474747')
fig.tight_layout(); plt.show()

## ⚡ Как переключить на GPU (обязательно для measured-результатов)

**По умолчанию этот ноутбук работает на бесплатном CPU (mock-профиль).**
Все числа mock — объявленная синтетика. Для measured-данных нужен GPU-прогон.

**Тяжёлые ожидания — в самом начале:** сразу после setup стоят `t07-gpu-bootstrap`
(ставит vLLM и поднимает сервер) и `t07-gpu-health` (ждёт готовности). Запустите
верхние клетки первыми — установка (~4 мин) и загрузка модели (~4 мин) пойдут,
пока вы разбираете теорию ручек. Финал ноутбука — реальный эксперимент:
default против вашего замороженного профиля на волнах судьи 32/64.

**Пошагово:**

1. **Runtime → Change runtime type → T4 GPU** (High-RAM выключить)
2. **Runtime → Restart session** (сбросить всё состояние)
3. Скопируйте и выполните ЭТУ клетку первой (перед t07-setup):
```python
import os
os.environ['LSM_T07_PROFILE'] = 'gpu'
print('GPU профиль активирован — далее Run all')
```
4. **Ctrl+F9 (Run all)**: bootstrap + health ~8 мин (идут фоном, пока вы на
   теории), мок-поиск ~2 мин, финальный эксперимент на реальном сервере
   ~6–12 мин (один перезапуск сервера с вашим профилем + волны 32/64)
5. После `t07-export` — **скачайте bundle и .ipynb** (File → Download) ДО отключения!

**Без шага 3 вы получите mock-числа, даже если GPU подключён.**
Если установка vLLM не удалась с первого раза (сетевой таймаут) — ноутбук
автоматически retry'ит до 3 раз. Если все 3 не удались — перезапустите
runtime и попробуйте ещё раз.

In [ ]:
import json, os, time, threading, hashlib, urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import TypedDict, Sequence
from tqdm.auto import tqdm  # правило курса: видимый прогресс в каждой клетке дольше ~5 с

PROFILE = os.environ.get('LSM_T07_PROFILE', os.environ.get('LSM_T06_PROFILE', 'mock'))  # 'mock' | 'gpu' (owner-run only)
ATTEMPT = 'attempt-001'
RESULTS = os.path.join('results', PROFILE, ATTEMPT)
os.makedirs(RESULTS, exist_ok=True)
print(f'profile={PROFILE}  results dir={RESULTS}')

In [ ]:
# GPU PROFILE ONLY. Pinned vLLM install + owned child service in one bootstrap.
# Стоит ПЕРВОЙ парой клеток: установка и загрузка модели идут, пока вы разбираете
# теорию ручек — к финальному эксперименту сервер уже горячий.
# Оценка времени: pip ~2–5 мин на попытку (тихо, флаг -q; до 3 попыток при сетевых
# сбоях), итог печатается ОДНОЙ строкой. Готовность сервера — следующая клетка.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import subprocess  # noqa: E402
    out = None
    _attempts = 0
    for _attempt in range(3):
        _attempts = _attempt + 1
        out = subprocess.run(['pip', 'install', '-q', 'vllm==0.30.0', '--timeout', '120'],
                             capture_output=True, text=True)
        if out.returncode == 0:
            break
        time.sleep(5)
    assert out.returncode == 0, ('vLLM install failed: ' + (out.stderr or '')[-300:])
    SERVER_CMD = ['python', '-m', 'vllm.entrypoints.openai.api_server',
                  '--model', 'Qwen/Qwen2.5-3B-Instruct',
                  '--revision', 'aa8e72537993ba99e69dfaafa59ed015b17504d1',
                  '--served-model-name', 'submission', '--dtype', 'float16',
                  '--max-model-len', '2048', '--port', '8017']
    server = subprocess.Popen(SERVER_CMD,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'w'),
                              stderr=subprocess.STDOUT)
    print(f'pip: vllm==0.30.0 установлен с попытки {_attempts}; '
          f'server pid {server.pid} — следующая клетка ждёт /health')

In [ ]:
# GPU PROFILE ONLY. Bounded readiness wait; never wave into a dead server.
# Оценка времени: обычно 60–180 с (загрузка весов 3B + CUDA-графы); потолок 900 с.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    import urllib.error  # noqa: E402
    waited = 0
    healthy = False
    deadline = time.monotonic() + 900
    with tqdm(total=900, desc='ждём /health', unit=' с', unit_scale=False) as bar:
        while not healthy:
            try:
                with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                    healthy = r.status == 200
            except (urllib.error.URLError, OSError):
                pass
            assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
            if not healthy:
                time.sleep(5); waited += 5; bar.update(5)
    print(f'service healthy; waited about {waited} s')

In [ ]:
class ModelConfig(TypedDict):
    """Serving-relevant model facts, declared once and shared by every calculation."""
    n_layers: int; n_kv_heads: int; head_dim: int; dtype_bytes: int

MODEL_FACTS: ModelConfig = dict(n_layers=36, n_kv_heads=2, head_dim=128, dtype_bytes=2)

class RequestRecord(TypedDict):
    """One client-observed request. Timestamps are client-side monotonic seconds."""
    request_id: str; wave: int; client_id: int; app_id: str; case_id: str
    input_tokens: int; max_tokens: int
    arrival_ts: float; first_byte_ts: float | None; terminal_ts: float
    output_tokens: int; http_status: int | None; outcome: str

class Percentiles(TypedDict):
    """Nearest-rank TTFT percentiles for one wave, completed requests only."""
    wave: int; count: int; p50_ms: float; p95_ms: float

class AuditSummary(TypedDict):
    """Damage classification of one wave's journal; nothing is repaired."""
    records: int; unique_ids: int; missing_terminal: int; duplicate_ids: int
    foreign_waves: int; impossible_time: int; complete: int

class Decision(TypedDict):
    """The chosen operating wave with its evidence and named binding constraint."""
    wave: int; throughput_tok_per_s: float; ttft_p95_ms: float; error_rate: float
    satisfies_slo: bool; reason: str

class InterferenceReport(TypedDict):
    """A-app degradation when B shares the server, plus B's completed count."""
    a_throughput_ratio: float; a_ttft_p95_ratio: float
    a_completed_alone: int; a_completed_shared: int; b_completed_shared: int

def save_jsonl(path: str, rows: Sequence[dict]) -> str:
    """Write a JSONL journal and return its sha256 hex digest.

    Parameters
    ----------
    path : str
        Destination file path, overwritten.
    rows : sequence of dict
        Records to serialize, one JSON object per line.

    Returns
    -------
    str
        sha256 of the written bytes, for provenance receipts.
    """
    with open(path, 'w', encoding='utf-8') as fh:
        for row in rows:
            fh.write(json.dumps(row, ensure_ascii=False) + chr(10))
    return hashlib.sha256(Path_bytes(path)).hexdigest()

def Path_bytes(path: str) -> bytes:
    """Read a file as raw bytes.

    Parameters
    ----------
    path : str
        File to read.

    Returns
    -------
    bytes
        Exact file content, for hashing.
    """
    with open(path, 'rb') as fh:
        return fh.read()

def load_jsonl(path: str) -> list[dict]:
    """Load a JSONL file into a list of dicts.

    Parameters
    ----------
    path : str
        Journal file, one JSON object per line.

    Returns
    -------
    list of dict
        Parsed records, in file order.
    """
    with open(path, encoding='utf-8') as fh:
        return [json.loads(line) for line in fh if line.strip()]

In [ ]:
# Данные, которыми занимается сервис: живые тексты замороженного корпуса.
# корпус — публичный файл курса (pinned-ревизия); качаем один раз и кэшируем рядом.
import pandas as pd
from IPython.display import display
PINNED_CORPUS_URL = (
    'https://raw.githubusercontent.com/levshaazz/'
    'llm-serving-mastery-submission-example/'
    '8525bd29959562db69800bf453555155f2ade668/classroom/topic06-purpose-v4/corpus.jsonl')
if not os.path.exists('corpus.jsonl'):
    urllib.request.urlretrieve(PINNED_CORPUS_URL, 'corpus.jsonl')
corpus_rows = [json.loads(l) for l in open('corpus.jsonl', encoding='utf-8') if l.strip()]
# берём 3 тикета A и 1 документ B — ровно то, чем будет нагружаться сервис
sample = ([r for r in corpus_rows if r['app_id'] == 'A'][:3]
          + [r for r in corpus_rows if r['app_id'] == 'B'][:1])
_tbl = pd.DataFrame([{'app': r['app_id'], 'case_id': r['case_id'], 'gold': r['gold'],
                      'tokens': r['input_tokens'],
                      'текст (начало)': r['user_text'][:80] + '…'} for r in sample])
display(_tbl.style.set_caption(
    f'всего {len(corpus_rows)} текстов замороженного корпуса;'
    ' A-тикет ≈ 100 токенов, B-документ ≈ 500; gold — правильная метка, не обещание модели'))

In [ ]:
# Та же нагрузка глазами длин: где живут A и B.
import matplotlib
try:
    get_ipython()  # интерактивный фронтенд (Colab/Jupyter): инлайн-бэкенд, графики видны
except NameError:
    matplotlib.use('Agg')  # headless-запуск: только файлы
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7, 3.2))
# две полупрозрачные гистограммы на общих осях: где живут длины A и B
for app, color in (('A', '#2563eb'), ('B', '#d97706')):
    lens = [r['input_tokens'] for r in corpus_rows if r['app_id'] == app]
    ax.hist(lens, bins=30, alpha=0.55, label=f'приложение {app} (n={len(lens)})', color=color)
ax.set_xlabel('длина входа, токены')
ax.set_ylabel('число текстов')
ax.set_title('Замороженный корпус: A-тикет против B-документа')
ax.legend(); fig.tight_layout(); plt.show()

## Блок 0 — Ручки vLLM: от механизма к параметру

В Topic 06 вы объясняли механизмы через картинку: batching экономит чтение весов,
KV-пул ограничивает число резидентов, prefill крадёт шаг у decode. Теперь у каждого
механизма есть именованная ручка — параметр запуска vLLM.

**Что вы должны увидеть на карте ниже (следующая клетка):** каждая ручка —
рычаг на механизме из Topic 06, и у каждой есть своя метрика на кривой.

**Ваша задача** — для каждой ручки написать:
- `expected_effect`: что произойдёт с кривой насыщения (одним-двумя предложениями)
- `risk`: чем вы жертвуете (одним предложением)

**Подсказки:**
- `--max-num-seqs 32`: меньше параллельных decode → каждый тик быстрее (меньше
  конкуренции за compute), но меньше запросов на batch → амортизация весов хуже
- `--gpu-memory-utilization 0.85`: меньше VRAM под KV → раньше наступает preemption,
  но больше места для активаций → меньше риск OOM
- `--max-model-len 1024`: scheduler планирует под короткие промпты → меньше
  зарезервированной памяти, но длинный промпт будет отклонён
- chunked prefill = True: prefill нового запроса делится на части → decode не
  останавливается на целый шаг → TPOT резидентов плавнее

Таблица **замораживается в файл ДО любых измерений** — как прогноз колена в Topic 06.
Когда измерение не совпадёт с предсказанием — это самые интересные данные, потому
что они указывают на механизм, который вы не учли.

In [ ]:
# Карта ручек Блока 0: ручка → механизм → что двигает на кривой (структура, не измерения).
import matplotlib.patches as mpatches
_rows = [
    ('--max-num-seqs', 'потолок batch decode', 'амортизация vs хвост'),
    ('--gpu-memory-utilization', 'доля VRAM под KV-пул', 'потолок резидентов'),
    ('--max-model-len', 'планирование worst-case', 'зарезервированная память'),
    ('chunked prefill', 'нарезка чужого prefill', 'TPOT резидентов'),
]
fig, ax = plt.subplots(figsize=(8.6, 3.6))
ax.axis('off'); ax.set_xlim(0, 10); ax.set_ylim(0, 4.6)
def _box(x: float, y: float, w: float, text: str, border: str) -> None:
    """Скруглённый бокс с текстом по центру (схема карты ручек).

    Parameters
    ----------
    x, y, w : float
        Левый нижний угол и ширина бокса в координатах осей.
    text : str
        Надпись по центру бокса.
    border : str
        Цвет рамки.

    Returns
    -------
    None
        Бокс добавляется на текущие оси; возвращаемого значения нет.
    """
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, 0.78,
                 boxstyle='round,pad=0.06', fc='white', ec=border, lw=1.4))
    ax.text(x + w / 2, y + 0.39, text, ha='center', va='center', fontsize=9.5)
for x, h in ((1.8, 'ручка'), (5.45, 'механизм из Topic 06'), (8.55, 'что двигает на кривой')):
    ax.text(x, 4.32, h, ha='center', fontsize=9, color='#5f6368')
for i, (knob, mech, metric) in enumerate(_rows):
    y = 3.35 - i * 0.94
    _box(0.25, y, 3.1, knob, '#2563eb')
    _box(4.0, y, 2.9, mech, '#0891b2')
    _box(7.35, y, 2.4, metric, '#d97706')
    ax.annotate('', xy=(3.93, y + 0.39), xytext=(3.42, y + 0.39),
                arrowprops=dict(arrowstyle='->', color='#5f6368'))
    ax.annotate('', xy=(7.28, y + 0.39), xytext=(6.97, y + 0.39),
                arrowprops=dict(arrowstyle='->', color='#5f6368'))
ax.set_title('Каждая ручка — рычаг на механизме, который вы уже объяснили')
fig.tight_layout(); plt.show()

In [ ]:
class MockTuningServer:
    """Расширенный мок с ручками vLLM: seqs, mem, chunked. Объявленная синтетика.

    max_num_seqs ограничивает эффективный параллелизм (эмулирует потолок batch).
    gpu_mem_util масштабирует ёмкость (меньше = раньше preemption).
    chunked разбивает prefill на шаги (decode не голодает один шаг).
    """
    def __init__(self, max_num_seqs: int = 256, gpu_mem_util: float = 0.9,
                 chunked: bool = False) -> None:
        """Bind the tuning knobs for this mock server instance.

        Parameters
        ----------
        max_num_seqs : int, default 256
            Ceiling on effective decode batch size.
        gpu_mem_util : float, default 0.9
            Fraction of VRAM available for the KV pool.
        chunked : bool, default False
            Whether large prefills are sliced across steps.

        Returns
        -------
        None
            Stateful helper; see request().
        """
        self.max_num_seqs = max_num_seqs
        self.gpu_mem_util = gpu_mem_util
        self.chunked = chunked
        self._lock = threading.Lock()
        self._in_flight = 0

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Serve one synthetic request under the configured knobs.

        Parameters
        ----------
        case : dict
            Load case with input_tokens.
        max_tokens : int
            Requested output tokens.

        Returns
        -------
        tuple of (str, int)
            Output text and output token count.
        """
        input_tokens = case['input_tokens']
        with self._lock:
            n = self._in_flight = self._in_flight + 1
        try:
            eff = min(n, self.max_num_seqs)
            queue_ms = 0.08 * eff * eff / max(0.5, self.gpu_mem_util)
            prefill_ms = 0.3 * input_tokens / (2.0 if self.chunked else 1.0)
            ttft = 0.15 + queue_ms / 1000 + prefill_ms / 1000
            per_tok = 0.002 * (1.0 + 0.06 * (eff - 1)) + 0.001 * 8.0 / max(1, eff)
            time.sleep(ttft)
            time.sleep(per_tok * max_tokens)
            return 'SYNTHETIC-MOCK-OUTPUT', max_tokens
        finally:
            with self._lock:
                self._in_flight -= 1

def run_wave(runner: object, app_id: str, cases: Sequence[dict], wave: int, per_client: int,
             timeout_s: float) -> list[RequestRecord]:
    """Run one closed-loop wave of `wave` clients, `per_client` requests each.

    Parameters
    ----------
    runner : object
        Service with .request(case, max_tokens); mock here, HTTP on gpu.
    app_id : str
        Application label written into every record.
    cases : sequence of dict
        Load cases with case_id and input_tokens (mock) or rendered (gpu).
    wave : int
        Concurrency: number of simultaneous closed-loop clients.
    per_client : int
        Sequential requests each client sends.
    timeout_s : float
        Per-request wall budget before the client gives up.

    Returns
    -------
    list of RequestRecord
        One record per terminated attempt, outcomes classified.
    """
    records: list[RequestRecord] = []
    lock = threading.Lock()

    def client(cid: int) -> None:
        """Send this client sequential requests and journal each attempt.

        Parameters
        ----------
        cid : int
            Client index inside the wave.

        Returns
        -------
        None
            Results are appended to the shared records list.
        """
        for seq in range(per_client):
            case = cases[(cid * per_client + seq) % len(cases)]
            rid = f'w{wave}-c{cid}-r{seq}'
            t0 = time.monotonic()
            outcome, status, out_tok, fb = 'connection_error', None, 0, None
            try:
                text, out_tok = runner.request(case, 32)
                fb = time.monotonic()
                outcome, status = ('completed', 200) if out_tok > 0 else ('truncated', 200)
                if time.monotonic() - t0 > timeout_s:
                    outcome = 'timeout'
            except TimeoutError:
                outcome = 'timeout'
            except Exception as exc:  # noqa: BLE001 - transport failures are data
                outcome = 'server_error' if '5xx' in str(exc) else 'connection_error'
            rec = RequestRecord(request_id=rid, wave=wave, client_id=cid, app_id=app_id,
                                case_id=case['case_id'], input_tokens=case['input_tokens'],
                                max_tokens=32, arrival_ts=t0, first_byte_ts=fb,
                                terminal_ts=time.monotonic(), output_tokens=out_tok,
                                http_status=status, outcome=outcome)
            if time.monotonic() - t0 > timeout_s and outcome == 'completed':
                rec['outcome'] = 'timeout'
            with lock:
                records.append(rec)

    with ThreadPoolExecutor(max_workers=wave) as pool:
        list(pool.map(client, range(wave)))
    return records


In [ ]:
def _wave_stats(records: Sequence[RequestRecord], w: int) -> dict:
    """Compute per-wave summary statistics for the table.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    w : int
        Wave label.

    Returns
    -------
    dict
        n, completed, timeouts, wall_s, throughput, p50_ms, p95_ms.
    """
    rs = [r for r in records if r['wave'] == w]
    done = [r for r in rs if r['outcome'] == 'completed']
    wall = max(r['terminal_ts'] for r in rs) - min(r['arrival_ts'] for r in rs)
    tt = sorted((r['first_byte_ts'] - r['arrival_ts']) * 1000
                for r in done if r['first_byte_ts'])
    return dict(wave=w, n=len(rs), completed=len(done),
                timeouts=sum(1 for r in rs if r['outcome'] == 'timeout'),
                wall_s=round(wall, 2),
                throughput=round(sum(r['output_tokens'] for r in done) / wall, 1),
                p50_ms=round(tt[len(tt) // 2]) if tt else None,
                p95_ms=round(tt[-(-95 * len(tt) // 100) - 1]) if tt else None)

def summary_rows(records: Sequence[RequestRecord], waves: Sequence[int]) -> 'pd.DataFrame':
    """Build the per-wave summary table as a DataFrame indexed by wave.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to include.

    Returns
    -------
    pandas.DataFrame
        One row per wave with outcomes and timing columns.
    """
    import pandas as pd
    return pd.DataFrame([_wave_stats(records, w) for w in waves]).set_index('wave')

def plot_curve(xs: Sequence[int], ys: Sequence[float], ylabel: str, path: str,
               title: str = '', slo_ms: float | None = None,
               knee: int | None = None,
               p50s: Sequence[float] | None = None) -> None:
    """One annotated curve: value labels, optional SLO line and knee marker.

    Parameters
    ----------
    xs : sequence of int
        Concurrency values.
    ys : sequence of float
        Metric values, same length as xs.
    ylabel : str
        Y-axis label with units.
    path : str
        File name inside RESULTS.
    title : str, default ''
        Optional figure title.
    slo_ms : float or None, default None
        SLO reference line in milliseconds.
    knee : int or None, default None
        Knee wave to annotate with an arrow.
    p50s : sequence of float or None, default None
        Optional second series (p50) drawn as a lighter line with a legend.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    fig, ax = plt.subplots(figsize=(7, 4.4))
    ax.plot(xs, ys, marker='o', color='#2563eb', label='p95')
    if p50s is not None:
        ax.plot(xs, p50s, marker='o', color='#93c5fd', label='p50')
        ax.legend(loc='center left')
    for i, (x, y) in enumerate(zip(xs, ys)):
        # p95-подпись сверху; при двух сериях p50-подпись снизу и только если линии разошлись
        ax.annotate(f'{y:.4g}', (x, y), textcoords='offset points', xytext=(0, 9),
                    ha='center', fontsize=9)
        if p50s is not None and ys[i] and p50s[i] / ys[i] < 0.82:
            ax.annotate(f'{p50s[i]:.4g}', (x, p50s[i]), textcoords='offset points',
                        xytext=(0, -14), ha='center', fontsize=8, color='#64748b')
    if slo_ms is not None:
        ax.axhline(slo_ms, color='#dc2626', linestyle='--', lw=1.4)
        ax.annotate(f'SLO {slo_ms:g}', xy=(1.0, slo_ms), xycoords=('axes fraction', 'data'),
                    ha='right', va='bottom', color='#dc2626', fontsize=9)
    if knee is not None and knee in list(xs):
        yk = ys[list(xs).index(knee)]
        ax.annotate('колено', (knee, yk), textcoords='offset points', xytext=(14, -20),
                    color='#15803d', fontsize=10,
                    arrowprops=dict(arrowstyle='->', color='#15803d'))
    ax.margins(x=0.10)
    ax.set_xlabel('конкурентность (закрытый цикл)')
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def outcome_shares(records: Sequence[RequestRecord], waves: Sequence[int], path: str) -> None:
    """Stacked per-wave outcome shares with percent labels; all records counted.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    waves : sequence of int
        Wave labels to plot.
    path : str
        File name inside RESULTS.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    kinds = ['completed', 'timeout', 'server_error', 'connection_error', 'truncated']
    fig, ax = plt.subplots(figsize=(6.5, 4))
    bottoms = [0.0] * len(waves)
    for kind in kinds:
        shares = []
        for w in waves:
            rs = [r for r in records if r['wave'] == w]
            shares.append(sum(1 for r in rs if r['outcome'] == kind) / len(rs) if rs else 0.0)
        ax.bar([str(w) for w in waves], shares, bottom=bottoms, label=kind)
        for i, (b, s) in enumerate(zip(bottoms, shares)):
            if s >= 0.06:
                ax.annotate(f'{s * 100:.0f}%', (i, b + s / 2), ha='center',
                            va='center', fontsize=8, color='white')
        bottoms = [b + s for b, s in zip(bottoms, shares)]
    ax.set_xlabel('конкурентность'); ax.set_ylabel('доля завершённых попыток')
    ax.set_title('Исходы по волнам (в знаменателе — каждая попытка)')
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc='upper left')
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

def plot_waterfall(records: Sequence[RequestRecord], wave: int, path: str,
                   max_rows: int = 40) -> None:
    """Request waterfall: grey wait, blue service, red timeout, one bar per request.

    Parameters
    ----------
    records : sequence of RequestRecord
        Full journal.
    wave : int
        Wave label to draw.
    path : str
        File name inside RESULTS.
    max_rows : int, default 40
        Cap on drawn requests, earliest arrivals first.

    Returns
    -------
    None
        Writes the PNG into RESULTS and shows it.
    """
    wave_rs = sorted((r for r in records if r['wave'] == wave),
                     key=lambda r: r['arrival_ts'])
    # равномерная выборка по времени прихода: видны и обслуженные, и таймауты
    step = max(1, len(wave_rs) // max_rows)
    rs = wave_rs[::step][:max_rows]
    t0 = wave_rs[0]['arrival_ts']
    timeout_budget = max((r['terminal_ts'] - r['arrival_ts']) for r in wave_rs
                         if r['outcome'] == 'timeout') if any(
        r['outcome'] == 'timeout' for r in wave_rs) else None
    fig, ax = plt.subplots(figsize=(9.5, max(3.6, 0.26 * len(rs))))
    for i, r in enumerate(rs):
        a = r['arrival_ts'] - t0
        if r['outcome'] == 'timeout' or r['first_byte_ts'] is None:
            ax.barh(i, r['terminal_ts'] - t0 - a, left=a, height=0.7, color='#dc2626')
        else:
            fb = r['first_byte_ts'] - t0
            ax.barh(i, fb - a, left=a, height=0.7, color='#cbd5e1')
            ax.barh(i, r['terminal_ts'] - t0 - fb, left=fb, height=0.7, color='#1d4ed8')
    if timeout_budget:
        ax.axvline(timeout_budget, color='#dc2626', linestyle=':', lw=1.6)
        ax.annotate('бюджет запроса: дальше клиент сдаётся', xy=(timeout_budget, len(rs) - 1),
                    xytext=(6, 0), textcoords='offset points', va='center',
                    color='#dc2626', fontsize=9)
    ax.set_ylim(-1.5, len(rs) + 0.5)
    ax.set_yticks([])
    ax.set_xlabel('секунды от начала волны', labelpad=8)
    n_to = sum(1 for r in wave_rs if r['outcome'] == 'timeout')
    ax.set_title(f'Один запрос = одна строка. Волна {wave}: всего {len(wave_rs)}, '
                 f'таймаутов {n_to}; показаны {len(rs)} равномерно по времени', pad=12)
    ax.text(0.99, 0.02, 'читать: светлый старт строки = ожидание в очереди; '
            'тёмно-синее = обслуживание (decode); красное до пунктира = не дожил',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=8.5,
            bbox=dict(boxstyle='round,pad=0.4', fc='#fffbeb', ec='#f59e0b', alpha=0.9))
    import matplotlib.patches as mpatches
    fig.legend(handles=[mpatches.Patch(color='#cbd5e1', label='ожидание (queue)'),
                        mpatches.Patch(color='#1d4ed8', label='обслуживание (decode)'),
                        mpatches.Patch(color='#dc2626', label='таймаут')],
               fontsize=8, ncol=3, loc='lower center', bbox_to_anchor=(0.5, -0.04))
    fig.tight_layout(rect=[0, 0.05, 1, 1])
    fig.savefig(os.path.join(RESULTS, path), dpi=120, bbox_inches='tight')
    plt.show()

In [ ]:
# GPU PROFILE ONLY. Frozen public corpus (pinned example revision), D/V splits.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    def _cases(app: str, split: str) -> list[dict]:
        """Select load cases for one application and split.

        Parameters
        ----------
        app : str
            Application label, 'A' or 'B'.
        split : str
            Corpus split, 'D' (development) or 'V' (held-out).

        Returns
        -------
        list of dict
            Cases with case_id, input_tokens and rendered prompt.
        """
        return [dict(case_id=r['case_id'], input_tokens=r['input_tokens'],
                     rendered=r['rendered_text'])
                for r in corpus_rows if r['split'] == split and r['app_id'] == app
                and r['length_bin'] == 'short']
    gpu_cases_A, gpu_cases_B = _cases('A', 'D'), _cases('B', 'D')
    heldout_A, heldout_B = _cases('A', 'V'), _cases('B', 'V')
    print(f'dev A={len(gpu_cases_A)} B={len(gpu_cases_B)}  '
          f'heldout A={len(heldout_A)} B={len(heldout_B)}')

In [ ]:
class HttpRunner:
    """Synchronous completions client for the local pinned service (case dict in)."""
    def __init__(self, port: int = 8017) -> None:
        """Point the client at one local service port.

        Parameters
        ----------
        port : int, default 8017
            Port of the owned vLLM service.

        Returns
        -------
        None
            Connectionless helper; see request().
        """
        self.url = f'http://127.0.0.1:{port}/v1/completions'

    def request(self, case: dict, max_tokens: int) -> tuple[str, int]:
        """Complete one rendered prompt.

        Parameters
        ----------
        case : dict
            Load case with the rendered prompt under 'rendered'.
        max_tokens : int
            Output token cap.

        Returns
        -------
        tuple of (str, int)
            Output text and completion token count.
        """
        body = json.dumps(dict(model='submission', prompt=case['rendered'],
                               max_tokens=max_tokens, temperature=0.0)).encode()
        req = urllib.request.Request(self.url, data=body,
                                     headers={'Content-Type': 'application/json'})
        with urllib.request.urlopen(req, timeout=120) as resp:
            data = json.loads(resp.read())
        return data['choices'][0]['text'], data['usage']['completion_tokens']

In [ ]:
def knob_effect_table() -> list[dict]:
    """Четыре ручки vLLM: механизм и ожидаемый эффект на кривой. Замораживается до измерений.

    Returns
    -------
    list of dict
        По строке на ручку: knob, mechanism, expected_effect, risk.
    """
    # Уровень A: ручки и механизмы вписаны — допишите expected_effect и risk.
    return [
        dict(knob='--max-num-seqs', mechanism='потолок batch decode', expected_effect='TODO', risk='TODO'),
        dict(knob='--gpu-memory-utilization', mechanism='доля VRAM под KV-пул', expected_effect='TODO', risk='TODO'),
        dict(knob='--max-model-len', mechanism='объявленный потолок контекста', expected_effect='TODO', risk='TODO'),
        dict(knob='chunked prefill', mechanism='нарезка чужого prefill', expected_effect='TODO', risk='TODO'),
    ]

_rows = knob_effect_table()
with open(os.path.join(RESULTS, 'knob-table.json'), 'w') as fh:
    json.dump(_rows, fh, ensure_ascii=False, indent=1)
display(pd.DataFrame(_rows).style.set_caption(
    f'таблица ручек заморожена до измерений ({len(_rows)} ручек, knob-table.json)'))

## Справочник: термины и команды блока

| Термин | Что это |
|---|---|
| knob | ручка vLLM: параметр запуска, который двигает кривую Topic 06 |
| OFAT | one-factor-at-a-time: меняем один фактор, остальные фиксированы |
| подтверждающая пара | конфигурация, совмещающая два эффекта — проверка взаимодействий |
| Парето-фронт | недоминируемые конфигурации: быстрее И/ИЛИ мягче всех остальных |
| SLO | граница продукта: p95 TTFT ≤ 800 мс, ошибки ≤ 1% |
| freeze (заморозка) | фиксация профиля + письменный прогноз ДО held-out |
| held-out | нагрузка V, которую профиль не видел: один прогон, один вердикт |

**Флаги vLLM, соответствующие ручкам:** `--max-num-seqs`, `--gpu-memory-utilization`,
`--enable-chunked-prefill`; `--max-model-len` объявлен фиксированным (2048) и в поиске
этой лабы не варьируется — он менял бы сам контракт, а не кривую.



## План поиска: предобъявленный и ограниченный

**Вопрос.** 4 ручки × 3 значения × 4 волны × 3 повтора — сотни GPU-часов.
Как уместить поиск в лабу и не обмануть себя?

**Метод: OFAT + подтверждающая пара — 5 конфигураций, ни больше ни меньше:**
1. **default** — базовая линия;
2. **3 OFAT** — каждая ручка по очереди от default, остальное не трогаем;
3. **1 пара** — два изменения вместе: проверка взаимодействия.

Каждая строка — гипотеза с механизмом («seqs-32: жертвую амортизацией ради хвоста»),
а не «попробуем и посмотрим». План замораживается ДО sweep (файл с timestamp и
sha256); изменение после seeing results — tuning, не validation; отвергнутые
строки остаются в таблице с причиной — как Triton-кандидат Round 03.

In [ ]:
def declare_search_space() -> list[dict]:
    """Объявить пространство поиска с бюджетом. Замораживается до измерений.

    Returns
    -------
    list of dict
        По строке на конфигурацию: name, max_num_seqs, gpu_mem_util, chunked, reason.
        Не более 6 конфигураций — бюджет ограничен.
    """
    # Уровень B: скелет готов — заполните поля для каждой конфигурации.
    # Шаг 1: включите default (max_num_seqs=256, mem=0.9, chunked=False).
    # Шаг 2: три ветки one-factor-at-a-time (seqs=32; mem=0.85; chunked=True)
    # и одна подтверждающая пара: seqs=32 + chunked=True.
    # Шаг 3: для каждой — одно предложение reason.
    raise NotImplementedError('TODO: заполните конфигурации по скелету')

_plan = declare_search_space()
with open(os.path.join(RESULTS, 'search-plan.json'), 'w') as fh:
    json.dump(_plan, fh, ensure_ascii=False, indent=1)
display(pd.DataFrame(_plan).style.set_caption(
    f'план поиска заморожен: {len(_plan)} конфигураций (search-plan.json)'))

In [ ]:
# Самопроверка плана ДО измерений: дисциплина поиска проверяется машиной.
assert len(_plan) <= 6, 'бюджет поиска: не более 6 конфигураций'
assert _plan[0]['name'] == 'default' and _plan[0]['max_num_seqs'] == 256, 'первая строка — default'
_base = _plan[0]
_multi = [r['name'] for r in _plan[1:]
          if sum([r['max_num_seqs'] != _base['max_num_seqs'],
                  r['gpu_mem_util'] != _base['gpu_mem_util'],
                  r['chunked'] != _base['chunked']]) > 1]
assert all(r.get('reason', '').strip() for r in _plan), 'у каждой конфигурации — reason'
assert len(_multi) == 1, f'ровно одна подтверждающая пара, найдено: {_multi or "нет"}'
print(f'план корректен: {_base["name"]} + {len(_plan) - 1 - len(_multi)} OFAT + '
      f'{len(_multi)} пара; бюджет и дисциплина соблюдены')

In [ ]:
# Лестница плана: высота ступени = сколько ручек изменено от default.
# Рисунок строится из ЗАМОРОЖЕННОГО _plan — это ваш план, а не иллюстрация из учебника.
import matplotlib.patches as mpatches

def _knob_delta(row: dict, base: dict) -> tuple[int, str]:
    """Число изменённых ручек и их список для строки плана против базовой.

    Parameters
    ----------
    row : dict
        Строка плана (max_num_seqs, gpu_mem_util, chunked).
    base : dict
        Базовая строка плана (default) для сравнения.

    Returns
    -------
    tuple of (int, str)
        Число изменённых ручек и подпись вида 'Δ seqs+prefill' ('' для default).
    """
    marks = []
    if row['max_num_seqs'] != base['max_num_seqs']:
        marks.append('seqs')
    if row['gpu_mem_util'] != base['gpu_mem_util']:
        marks.append('mem')
    if row['chunked'] != base['chunked']:
        marks.append('prefill')
    return len(marks), ('Δ ' + '+'.join(marks) if marks else 'baseline')

fig, ax = plt.subplots(figsize=(8.4, 3.4))
ax.axis('off'); ax.set_xlim(-0.5, len(_plan) + 1.7); ax.set_ylim(-0.75, 3.0)
for i, row in enumerate(_plan):
    height, delta = _knob_delta(row, _base)
    ax.add_patch(mpatches.FancyBboxPatch((i - 0.42, height - 0.32), 0.84, 0.7,
                 boxstyle='round,pad=0.03',
                 fc='#f8fafc' if height else '#eef2ff',
                 ec='#2563eb' if height == 1 else ('#16a34a' if height == 2 else '#94a3b8'),
                 lw=1.5))
    ax.text(i, height + 0.07, row['name'], ha='center', va='center',
            fontsize=9.5, family='monospace')
    ax.text(i, height - 0.17, delta, ha='center', va='center',
            fontsize=8.2, color='#474747')
    if i:
        prev_h = _knob_delta(_plan[i - 1], _base)[0]
        ax.plot([i - 1 + 0.45, i - 0.45], [prev_h, prev_h], color='#94a3b8', lw=1.2, ls=':')
        ax.plot([i - 0.45, i - 0.45], [prev_h, height], color='#94a3b8', lw=1.2, ls=':')
ax.text(len(_plan) + 0.15, 2.0, 'пара (2 ручки)', fontsize=8.5, color='#16a34a', va='center')
ax.text(len(_plan) + 0.15, 1.0, 'OFAT (1 ручка)', fontsize=8.5, color='#2563eb', va='center')
ax.text(len(_plan) + 0.15, 0.0, 'базовая линия', fontsize=8.5, color='#64748b', va='center')
ax.set_title(f'Лестница плана: {len(_plan)} конфигураций, заморожено ДО sweep')
fig.tight_layout(); plt.show()

In [ ]:
# Константы волн и нагрузка для мок-профиля (перенесены из удалённой t07-run-waves).
WAVES = [8, 32, 64, 128]
MOCK_CASES = [dict(case_id=f'synth-{i}', input_tokens=n)
              for i, n in enumerate([100, 110, 120] * 40)]

# Исполнение поиска: мок-профиль (и на CPU, и на GPU — реальный сервер в поиске
# не участвует). Оценка времени: ~1–2 мин. measured-числа даёт только
# финальный эксперимент в конце ноутбука.
# Неудачные конфигурации — строки таблицы, а не вычёркнутые сноски.
sweep_rows = []
for cfg in tqdm(_plan, desc='sweep', unit='cfg'):
    mock = MockTuningServer(max_num_seqs=cfg['max_num_seqs'],
                            gpu_mem_util=cfg['gpu_mem_util'],
                            chunked=cfg['chunked'])
    recs = []
    for w in tqdm(WAVES, desc=f"  {cfg['name']}", unit=' волна', leave=False):
        recs.extend(run_wave(mock, 'A', MOCK_CASES, w, per_client=3, timeout_s=12.0))
    stats = {w: _wave_stats(recs, w) for w in WAVES}
    sweep_rows.append(dict(
        name=cfg['name'], seqs=cfg['max_num_seqs'], mem=cfg['gpu_mem_util'],
        chunked=cfg['chunked'],
        thr_64=stats[64]['throughput'],
        p95_64=stats[64]['p95_ms'] if stats[64]['p95_ms'] is not None else 99999,
        err_64=stats[64]['timeouts'] / max(1, stats[64]['n']),
    ))
    with open(os.path.join(RESULTS, 'raw-journal.jsonl'), 'a') as jf:
        for rec in recs:
            jf.write(json.dumps(rec, ensure_ascii=False) + chr(10))

# Траектория свипа: те же точки в ПОРЯДКЕ запуска — как поиск прошёл по пространству.
_fig, _ax = plt.subplots(figsize=(8.0, 4.2))
_ax.plot([r['p95_64'] for r in sweep_rows], [r['thr_64'] for r in sweep_rows],
         ':', color='#94a3b8', lw=1.4, zorder=1)
for i, r in enumerate(sweep_rows):
    _ax.scatter(r['p95_64'], r['thr_64'], s=110, zorder=3,
                color='#2563eb' if i else '#64748b')
    _ax.annotate(str(i + 1), (r['p95_64'], r['thr_64']), ha='center', va='center',
                 fontsize=8.5, color='white', zorder=4)
    _ax.annotate(r['name'], (r['p95_64'], r['thr_64']), xytext=(6, -16),
                 textcoords='offset points', fontsize=8.5, family='monospace')
_ax.axvline(800, color='#dc2626', ls='--', lw=1.4)
_ax.annotate('SLO 800 мс', (800, max(r['thr_64'] for r in sweep_rows)),
             xytext=(-6, -14), textcoords='offset points', ha='right',
             fontsize=9, color='#dc2626')
_ax.set_xlabel('p95 TTFT при волне 64, мс — меньше лучше')
_ax.set_ylabel('throughput, ток/с — больше лучше')
_ax.set_title('Траектория свипа: номера = порядок запуска конфигураций')
_fig.tight_layout(); plt.show()

In [ ]:
# Таблица поиска — «честная таблица»: неудачные конфигурации остаются строками.
_table = pd.DataFrame(sweep_rows)
_table.to_json(os.path.join(RESULTS, 'sweep-table.json'), orient='records', force_ascii=False, indent=1)
display(_table.style.set_caption('sweep завершён: конфигурация → скорость и хвост (sweep-table.json)'))

## Парето-фронт: скорость против хвоста

**Вопрос.** Каждая конфигурация — точка «throughput (больше — лучше) × p95 TTFT
(меньше — лучше)». Как выбирать, если одна быстрее, а другая мягче?

**Доминирование:** A доминирует B, если A не медленнее И не грубее, и хотя бы по
одному — строго. Фронт = недоминируемые точки; на фронте нет «лучшего» — есть
trade-off, который разрешает SLO. На графике: синие ромбы — фронт, красные —
доминируемые, зелёная зона слева от SLO 800 мс — eligible.

**Читать так:** среди eligible взять точку с максимальным throughput; проверить,
есть ли на фронте точка чуть медленнее, но значительно мягче — это запас
прочности, а не потеря.

**Вопрос себе:** если SLO станет 600 мс, какие конфигурации останутся eligible —
и что это говорит о хрупкости выбора?

In [ ]:
def pareto_front(rows: Sequence[dict]) -> list[dict]:
    """Отобрать Парето-оптимальные конфигурации: скорость max, хвост min.

    Parameters
    ----------
    rows : sequence of dict
        Строки sweep-таблицы с thr_64 и p95_64.

    Returns
    -------
    list of dict
        Только те строки, где ни одна другая не быстрее И не короче одновременно.
    """
    raise NotImplementedError('TODO: доминирование = быть лучше по обоим критериям')

_front = pareto_front(sweep_rows)
display(pd.DataFrame(_front))

In [ ]:
# Паретовский фронт одной картинкой: скорость против хвоста.
fig, ax = plt.subplots(figsize=(7, 4.5))
for row in sweep_rows:
    color = '#dc2626' if row not in _front else '#2563eb'
    marker = 'o' if row not in _front else 'D'
    ax.scatter(row['p95_64'], row['thr_64'], c=color, marker=marker, s=90, zorder=3)
    ax.annotate(row['name'], (row['p95_64'], row['thr_64']),
                textcoords='offset points', xytext=(8, 4), fontsize=9)
ax.set_xlabel('p95 TTFT на волне 64, мс (меньше — лучше →)')
ax.set_ylabel('throughput на волне 64, ток/с')
ax.axvline(800, color='#dc2626', linestyle='--', lw=1.4)
ax.annotate('SLO p95 800 мс', (800, ax.get_ylim()[0]), xytext=(6, 12),
            textcoords='offset points', color='#dc2626', fontsize=9)
ax.set_title('Паретовский фронт: скорость против хвоста (синие ромбы)')
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
# Связка с Topic 06: что выбор на фронте означает против кривой насыщения.
_default = next(r for r in sweep_rows if r['name'] == 'default')
_gain = round(_front[0]['thr_64'] / _default['thr_64'], 2) if _front else None
print(f'front против default: x{_gain} по throughput на волне 64; '
      'вспомните колено из Topic 06 — прирост живёт ДО колена, за ним начинаются отказы')

## Заморозка профиля: финальный выбор и его последствия

**Вопрос.** Как закончить поиск так, чтобы результат нельзя было подогнать?

Среди eligible на фронте — самая быстрая; в файл с timestamp; письменный прогноз
held-out ДО его запуска. Дальше профиль не меняется: проваленный held-out —
честный reject, а не повод перевыбрать (изменение после увиденного — tuning,
не validation).

**Три честных вердикта:** **accept** — SLO держится на новых текстах, прогноз
сработал; **reject** — продукт требует меньшего обещания или большего железа
(сказать это с доказательствами — работа инженера); **inconclusive** — «здоров»
не отличить от «повезло»: ещё один прогон, а не надежда в memo.

**Мемо читают три стороны:** продакт — «что обещаем», capacity — «где кривая
гнётся», скептик — «какое объяснение этот прогон не исключает». Все три вердикта —
полноценные результаты: оценивается заморозка, не удача.

In [ ]:
# Заморозка производственного профиля под SLO: p95 ≤ 800 мс, ошибки ≤ 1%.
_slo = dict(ttft_p95_ms=800.0, max_error_rate=0.01)
_eligible = [r for r in _front if r['p95_64'] <= _slo['ttft_p95_ms'] and r['err_64'] <= _slo['max_error_rate']]
chosen = max(_eligible, key=lambda r: r['thr_64']) if _eligible else None
frozen_name = chosen['name'] if chosen else 'NONE'
from IPython.display import HTML
if chosen is not None:  # карточка профиля — единственная эмиссия клетки
    display(HTML(
        '<div style="border:2px solid #188038;border-radius:10px;padding:10px 14px;'
        'max-width:680px;font-family:sans-serif;line-height:1.55">'
        '<div style="font-size:15px">❄ <b>Замороженный профиль:</b> ' + chosen['name'] + '</div>'
        '<div>thr@64 = ' + f"{chosen['thr_64']:.1f}" + ' ток/с · p95@64 = '
        + f"{chosen['p95_64']:.0f}" + ' мс (SLO ≤ 800 мс) · err = '
        + f"{chosen['err_64']:.3f}" + ' (≤ 0.01)</div>'
        '<div style="color:#5f6368">прогноз — в следующей клетке; политика не меняется '
        'до вердикта held-out</div></div>'))
else:
    display(HTML('<div style="font-size:15px;color:#c5221f">❄ <b>Замораживать нечего:</b> '
                 'ни один профиль не прошёл SLO — финал темы остановится на REJECT.</div>'))

In [ ]:
def author_forecast(name: str) -> str:
    """Письменный прогноз held-out для замороженного профиля — ДО прогона.

    Parameters
    ----------
    name : str
        Имя замороженной конфигурации.

    Returns
    -------
    str
        Прогноз с числами: ожидаемый throughput и p95 на нагрузке V.
    """
    raise NotImplementedError('TODO: прогноз должен быть фальсифицируемым — назовите числа')
    # Скелет: имя профиля, ожидаемый thr на V (и вилка, которую считаем нормой),
    # ожидаемый p95 против SLO 800 мс, что считаете опровержением.

_forecast = author_forecast(frozen_name)
with open(os.path.join(RESULTS, 'profile.json'), 'w') as fh:
    json.dump(dict(name=frozen_name, forecast=_forecast, slo=_slo,
                   frozen_at=time.time()), fh, ensure_ascii=False, indent=1)
print(f'профиль {frozen_name} заморожен с прогнозом — политика не меняется до вердикта')

In [ ]:
# Held-out: ДРУГАЯ нагрузка (V), тот же замороженный профиль — один прогон.
# Mock-профиль: синтетический V-набор с ДРУГИМ seed и профилем длин (объявлено).
# GPU-профиль: настоящие тексты V-сплита корпуса (heldout_A из t07-gpu-cases).
if chosen is None:
    print('ни один профиль не прошёл SLO — вердикт REJECT: продукт требует '
           'меньшего обещания или большей машины; held-out не выполняется')
else:
    if PROFILE == 'gpu':
        v_runner, v_cases = HttpRunner(), heldout_A
        v_note = 'нагрузка V: настоящие тексты V-сплита корпуса'
    else:
        import random as _random
        _rng = _random.Random(20261026)
        v_cases = [dict(case_id=f'v-{i}', input_tokens=_rng.choice((88, 132, 104, 146)))
                   for i in range(len(MOCK_CASES))]
        v_runner = MockTuningServer(max_num_seqs=chosen['seqs'],
                                    gpu_mem_util=chosen['mem'], chunked=chosen['chunked'])
        v_note = 'нагрузка V: синтетика с другим seed и профилем длин (объявлено)'
    heldout_recs = run_wave(v_runner, 'A', v_cases, 64, per_client=3, timeout_s=12.0)
    held_stats = _wave_stats(heldout_recs, 64)
    with open(os.path.join(RESULTS, 'heldout.json'), 'w') as fh:
        json.dump(dict(note=v_note, stats=held_stats), fh, ensure_ascii=False, indent=1)
    _ok = held_stats['p95_ms'] <= 800  # сводка против SLO — единственная эмиссия клетки
    from IPython.display import HTML
    display(HTML(
        '<div style="font-size:14.5px;line-height:1.6">'
        '<b>held-out</b> (' + v_note + '): thr=' + str(held_stats['throughput'])
        + ', p95=' + str(held_stats['p95_ms']) + ' мс, timeouts='
        + str(held_stats['timeouts']) + '/' + str(held_stats['n']) + '<br>'
        + ('✅' if _ok else '❌') + ' против SLO 800 мс — '
        + ('внутри бюджета' if _ok else 'вне бюджета')
        + '. Итоговый вердикт (против прогноза) — в memo, следующая клетка.</div>'))

In [ ]:
def write_memo(verdict: str, reason: str) -> dict:
    """Мемо решения: вердикт, причина, обещание — три обязательные строки.

    Parameters
    ----------
    verdict : str
        accept, reject или inconclusive — по числам held-out против прогноза.
    reason : str
        Одна фраза: какая цифра прогноза сбылась или нарушилась.

    Returns
    -------
    dict
        Записанный мемо (memo.json).
    """
    raise NotImplementedError('TODO: вердикт по цифрам held-out против прогноза')

_memo = write_memo('TODO-verdict', 'TODO-reason')
print('мемо записан:', _memo)

## Stretch: краевой риск и план отката

**Вопрос.** Что сломается первым при росте нагрузки или длины промптов — до того,
как это случится в продакшене?

Краевой тест: длиннейший промпт корпуса (817 токенов) против наименьшего KV-пула
(mem=0.70) — обслуживается или отклоняется?

**План отката — операционный артефакт:** **trigger** — метрика, порог, длительность
(«p95 > 800 мс в течение 5 минут», а не «станет медленно»); **action** — revert to
default / halve load / page on-call; **evidence** — какой дашборд проверит дежурный
в 3 ночи. План без триггера — надежда, без действия — наблюдение, без evidence —
неисполним.

In [ ]:
# Stretch: краевой тест — длиннейший промпт против наименьшего KV-пула.
_edge_case = dict(case_id='edge-long-prompt', input_tokens=817)
_edge_server = MockTuningServer(max_num_seqs=32, gpu_mem_util=0.70, chunked=True)
try:
    _edge_t, _edge_n = _edge_server.request(_edge_case, 32)
    print(f"краевой тест: длинный промпт ОБСЛУЖЕН ({_edge_n} токенов) при mem=0.70")
except Exception as e:
    print(f"краевой тест: длинный промпт ОТКЛОНЁН при mem=0.70: {type(e).__name__}")

In [ ]:
# Stretch: план отката — условие возврата к default.
rollback_trigger = 'TODO: метрика + порог + длительность (например: p95 > 800ms в течение 5 минут)'
rollback_action = 'TODO: действие при триггере (revert to default / halve load / page on-call)'
rollback_evidence = 'TODO: какой дашборд/запрос/порог проверяет дежурный в 3 ночи'
with open(os.path.join(RESULTS, 'rollback-plan.json'), 'w') as fh:
    json.dump(dict(trigger=rollback_trigger, action=rollback_action,
                   evidence=rollback_evidence), fh, ensure_ascii=False, indent=1)
print('план отката записан — заполните три TODO выше')

## Финальный эксперимент: ваш сервер против default

**Вопрос.** Весь поиск шёл по мок-модели. Что даст тот же конвейер на живом сервисе?

Судья раунда: throughput на волнах 32 и 64 (корнем из произведения), p95 против
SLO, доля ошибок. Две конфигурации — `default` и замороженный профиль — по две
волны судьи, один драйвер, один журнал. На GPU сервер перезапускается с ручками
профиля (vLLM принимает `--max-num-seqs` только при старте); на mock тот же
конвейер идёт через MockTuningServer — числа синтетические, ход анализа тот же.

**Вопрос на защиту:** прирост judge-метрики — это перевод ручки в механизм или
заслуга измерения? Что должен показать повторный прогон (вспомните флип колена
64 → 32 → 64 из Темы 6)?

In [ ]:
# Финальный эксперимент: default против замороженного профиля, волны судьи 32/64.
# Оценка времени: mock ~30 с; GPU ~6–12 мин (один перезапуск сервера + 4 волны).
import math  # noqa: E402

def _gpu_apply_config(cfg: dict | None) -> None:
    """GPU ONLY: перезапустить owned-сервис с профилем (None = default-аргументы).

    Parameters
    ----------
    cfg : dict or None
        Строка профиля с seqs/mem/chunked; None — базовая команда без ручек.

    Returns
    -------
    None
        Перепривязывает глобальный server и ждёт /health (потолок 900 с).
    """
    global server
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    cmd = list(SERVER_CMD)
    if cfg is not None:
        cmd += ['--max-num-seqs', str(cfg['seqs']),
                '--gpu-memory-utilization', str(cfg['mem'])]
        if cfg['chunked']:
            cmd.append('--enable-chunked-prefill')
    server = subprocess.Popen(cmd,
                              stdout=open(os.path.join(RESULTS, 'server.log'), 'a'),
                              stderr=subprocess.STDOUT)
    deadline = time.monotonic() + 900
    with tqdm(total=900, desc=f"health: {cfg['name'] if cfg else 'default'}", unit=' с') as bar:
        while True:
            try:
                with urllib.request.urlopen('http://127.0.0.1:8017/health', timeout=5) as r:
                    if r.status == 200:
                        break
            except OSError:
                pass
            assert time.monotonic() < deadline, 'service did not become healthy in 900 s'
            time.sleep(5)
            bar.update(5)

def judge_geomean(row: dict) -> float:
    """Оценка судьи раунда: корень из произведения throughput на волнах 32 и 64.

    Parameters
    ----------
    row : dict
        Строка финала с thr_32 и thr_64, tok/s.

    Returns
    -------
    float
        Геометрическое среднее throughput, tok/s.
    """
    # Уровень A: клетки выше уже прогнали обе конфигурации — объясните соседу,
    # почему судья берёт именно корень из произведения, а не среднее арифметическое.
    # Уровень B: реализуйте judge-метрику (math уже импортирован).
    # Уровень C: добавьте в _final_cfgs третьей строкой второй SLO-eligible
    # профиль из _front и одной фразой объясните, что меняет его присутствие.
    raise NotImplementedError('TODO: корень из произведения thr_32 * thr_64')

if chosen is None:
    print('ни один профиль не прошёл SLO — финальный эксперимент пропущен (вердикт REJECT)')
else:
    _final_rows = []
    _final_cfgs = [dict(name='default', seqs=256, mem=0.9, chunked=False), chosen]
    for cfg in tqdm(_final_cfgs, desc='финал', unit=' конфиг'):
        if PROFILE == 'gpu':
            _gpu_apply_config(None if cfg['name'] == 'default' else cfg)
            runner, cases = HttpRunner(), gpu_cases_A
        else:
            runner = MockTuningServer(max_num_seqs=cfg['seqs'],
                                      gpu_mem_util=cfg['mem'], chunked=cfg['chunked'])
            cases = MOCK_CASES
        recs = []
        for w in tqdm((32, 64), desc=f"  {cfg['name']}", unit=' волна', leave=False):
            recs.extend(run_wave(runner, 'A', cases, w, per_client=3, timeout_s=20.0))
        stats = {w: _wave_stats(recs, w) for w in (32, 64)}
        _final_rows.append(dict(
            name=cfg['name'], seqs=cfg['seqs'], mem=cfg['mem'], chunked=cfg['chunked'],
            thr_32=stats[32]['throughput'], thr_64=stats[64]['throughput'],
            p95_64=stats[64]['p95_ms'] if stats[64]['p95_ms'] is not None else 99999,
            err_64=stats[64]['timeouts'] / max(1, stats[64]['n'])))
    for row in _final_rows:
        row['judge_tok_s'] = judge_geomean(row)
    _gain = _final_rows[1]['judge_tok_s'] / _final_rows[0]['judge_tok_s']
    _slo_ok = _final_rows[1]['p95_64'] <= 800 and _final_rows[1]['err_64'] <= 0.01
    with open(os.path.join(RESULTS, 'final-experiment.json'), 'w') as fh:
        json.dump(dict(rows=_final_rows, gain=round(_gain, 3), slo_ok=_slo_ok,
                       profile_run=PROFILE), fh, ensure_ascii=False, indent=1)
    display(pd.DataFrame(_final_rows).style.set_caption(
        f'{"MOCK" if PROFILE == "mock" else "GPU"}: прирост judge-метрики '
        f'x{_gain:.2f}; SLO-вердикт замороженного профиля: '
        f'{"PASS" if _slo_ok else "FAIL"} (final-experiment.json)'))

## Что вы построили — и чего этот прогон НЕ доказывает

**Артефакт:** замороженный профиль с прогнозом, подтверждённый (или опровергнутый)
held-out'ом, план отката с триггером/действием/evidence и финальный эксперимент
default-против-профиля с judge-метрикой — всё в results/ с sha256-хэшами.

**Чему научились:** ручка → механизм; план ДО измерений; чтение Парето-фронта;
заморозка с фальсифицируемым прогнозом; held-out с тремя вердиктами; план отката,
исполнимый в 3 ночи; выигрыш против default на живом сервисе.

**Чего НЕ доказывает:** mock — объявленная синтетика (формула написана в коде,
а не открыта измерением); один прогон — не статистика; burst-волны по 3 запроса
на клиента — не устоявшаяся нагрузка; learner validation не проводилась.
measured-числа — только GPU-профиль.

**Дальше:** Topic 08 — производственный API поверх профиля: cancellation, quotas,
backpressure. Профиль, который вы научились строить здесь, вы научитесь защищать там.

In [ ]:
artifacts = {}
for name in ('knob-table.json', 'search-plan.json', 'sweep-table.json', 'profile.json',
             'raw-journal.jsonl', 'heldout.json', 'memo.json', 'final-experiment.json'):
    p = os.path.join(RESULTS, name)
    if os.path.exists(p):
        artifacts[name] = hashlib.sha256(Path_bytes(p)).hexdigest()
with open(os.path.join(RESULTS, 'manifest.json'), 'w') as fh:
    json.dump(dict(profile=PROFILE, artifacts=artifacts), fh, ensure_ascii=False, indent=1)
import shutil  # noqa: E402
shutil.make_archive('/content/t07-' + PROFILE + '-bundle', 'gztar', RESULTS)
print(f'{len(artifacts)} артефактов хэшировано; profile={PROFILE}; '
      f'bundle: /content/t07-{PROFILE}-bundle.tar.gz')

In [ ]:
# GPU PROFILE ONLY. Owned teardown of the service started in t07-gpu-bootstrap.
if PROFILE != 'gpu':
    print('skipped: gpu-only cell (profile is mock)')
else:
    server.terminate()
    try:
        server.wait(timeout=30)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()
    print('server exit code', server.returncode,
          '- delete the runtime only after the bundle is downloaded')